# Train DenseTwoTower

In [1]:
# ! pip install "pandas<=2.3.2" "numpy" "torch<=2.5" "matplotlib" "seaborn" "matplotlib-venn" "datasets" "ipykernel" "recbole" "kmeans-pytorch" "sentence-transformers"

In [2]:
import numpy as np

# For NumPy 2.0 compatibility with RecBole 1.2
np.float_ = np.float64
np.int_ = np.int64
np.complex_ = np.complex128
np.unicode_ = np.str_

# Ensure logging on notebook works even on Colab
import logging
logging.getLogger().handlers.clear()

In [3]:
from typing import Any
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from recbole.config import Config
from recbole.data.dataloader import FullSortEvalDataLoader, AbstractDataLoader
from recbole.data import create_dataset, data_preparation
from recbole.model.abstract_recommender import GeneralRecommender
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger, InputType, ModelType
from sentence_transformers import SentenceTransformer

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# --- Config ---
# Assume we have `*.train.inter`, `*.valid.inter`, `*.test.inter`
DATASET_NAME: str = "beauty"
DATA_DIR: str = "../data"
SEED = 67
DEVICE = "mps" # Other options: "cpu", "cuda"

## Create dataset

In [5]:
config_dict: dict[str, Any] = {
    "data_path": DATA_DIR,
    "dataset": DATASET_NAME,
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "benchmark_filename": ["train", "valid", "test"],
    "load_col": {
        "inter": ["user_id", "item_id"],
        "user": ["user_id", "category"],
        "item": ["item_id", "title"],
    },
    "epochs": 100,
    "train_batch_size": 4096,
    "eval_batch_size": 409_600_000,
    # Must not use negative sampling; we use in-batch negatives in InfoNCE
    "train_neg_sample_args": None,
    "eval_args": {
        # Split is already determined by the `benchmark filename` as separate `.inter` files
        "split": None,
        "order": "TO",
        "mode": {"valid": "full", "test": "full"},
    },
    "metrics": ["NDCG", "Recall", "MRR"],
    "topk": [20],
    "valid_metric": "NDCG@20",
    "seed": SEED,
    "learning_rate": 0.001 
}

config: Config = Config(model="Pop", config_dict=config_dict)
config.final_config_dict["device"] = torch.device(DEVICE)

init_logger(config)
init_seed(SEED, reproducibility=True)

In [6]:
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
21 Jun 22:19    INFO  [Training]: train_batch_size = [4096] train_neg_sample_args: [{'distribution': 'none', 'sample_num': 'none', 'alpha': 'none', 'dynamic': False, 'candidate_num': 0}]
21 Jun 22:19    INFO  [Evaluation]: eval_batch_size = [409600000] eval_args: [{'split': None, 'order': 'TO', 'group_by': 'user', 'mode': {'va

## Define DenseTwoTower

In [7]:
class DenseTwoTower(GeneralRecommender):
    input_type = InputType.POINTWISE
    type = ModelType.TRADITIONAL

    def __init__(self, config, dataset):
        super().__init__(config, dataset)

        self.tau = 0.1
        self.embed_size = 128

        # ── Step 1: Get title strings via RecBole's data pipeline ──
        title_tokens: torch.Tensor = dataset.item_feat["title"]
        id2token: dict[int, str] = {
            v: k for k, v in dataset.field2token_id["title"].items()
        }
        titles: list[str] = []
        for tok in title_tokens:
            t = id2token.get(tok.item(), "")
            titles.append(t)

        # ── Step 2: Encode with BGE text encoder ──
        self.logger.info(
            "Encoding %d item titles with BGE (BAAI/bge-base-en-v1.5)...", self.n_items
        )
        bge_model = SentenceTransformer("BAAI/bge-base-en-v1.5", device=config["device"])
        embs = bge_model.encode(titles, show_progress_bar=True, batch_size=256, normalize_embeddings=True)
        embs = torch.from_numpy(embs).float()

        # Zero out padding item (index 0) so it never gets recommended
        with torch.no_grad():
            embs[0] = 0.0

        self.register_buffer("item_embeddings", embs)

        # ── Step 3: User tower (learned) ──
        self.user_embedding = nn.Embedding(self.n_users, self.embed_size)
        nn.init.normal_(self.user_embedding.weight, std=0.01)

        self.user_mlp = nn.Sequential(
            nn.Linear(self.embed_size, self.embed_size),
            nn.GELU(),
            nn.Linear(self.embed_size, self.embed_size),
        )

        # ── Step 4: Item projection MLP (trainable, applied to frozen BGE embeddings) ──
        self.item_proj = nn.Sequential(
            nn.Linear(768, 512),
            nn.GELU(),
            nn.Linear(512, self.embed_size),
        )

        # Caches for full-sort evaluation (avoid recomputing every batch)
        self._cached_user_emb: torch.Tensor | None = None
        self._cached_item_emb: torch.Tensor | None = None

    def calculate_loss(self, interaction):
        # Clear cached embeddings when training starts (weights may have changed)
        if self._cached_user_emb is not None or self._cached_item_emb is not None:
            self._cached_user_emb = self._cached_item_emb = None

        users = interaction[self.USER_ID]
        items = interaction[self.ITEM_ID]

        # User tower: lookup → MLP → normalize
        u_emb = self.user_embedding(users)
        u_emb = self.user_mlp(u_emb)
        u_emb = F.normalize(u_emb, dim=-1)

        # Item tower: frozen BGE → projection → normalize
        i_emb = self.item_proj(self.item_embeddings[items])
        i_emb = F.normalize(i_emb, dim=-1)

        # InfoNCE loss with in-batch negatives
        logits = torch.matmul(u_emb, i_emb.transpose(0, 1)) / self.tau
        targets = torch.arange(len(users), device=logits.device)

        return F.cross_entropy(logits, targets)

    def predict(self, interaction):
        users = interaction[self.USER_ID]
        items = interaction[self.ITEM_ID]

        u_emb = F.normalize(self.user_mlp(self.user_embedding(users)), dim=-1)
        i_emb = F.normalize(self.item_proj(self.item_embeddings[items]), dim=-1)

        return torch.sum(u_emb * i_emb, dim=-1)

    @torch.no_grad()
    def full_sort_predict(self, interaction):
        users = interaction[self.USER_ID]

        if self._cached_user_emb is None or self._cached_item_emb is None:
            self._cached_user_emb = F.normalize(
                self.user_mlp(self.user_embedding.weight), dim=-1
            )
            self._cached_item_emb = F.normalize(
                self.item_proj(self.item_embeddings), dim=-1
            )

        batch_users = self._cached_user_emb[users]
        return torch.matmul(batch_users, self._cached_item_emb.t())

## Train DenseTwoTower

In [8]:
# Suppress httpx INFO logs from SentenceTransformer's model loading
logging.getLogger('httpx').setLevel(logging.WARNING)

model: DenseTwoTower = DenseTwoTower(config, train_data.dataset).to(config["device"])
trainer: Trainer = Trainer(config, model)

best_valid_score, best_valid_result = trainer.fit(train_data, valid_data)

21 Jun 22:19    INFO  Encoding 250853 item titles with BGE (BAAI/bge-base-en-v1.5)...
21 Jun 22:19    INFO  Loading SentenceTransformer model from BAAI/bge-base-en-v1.5.
Batches: 100%|██████████| 980/980 [06:58<00:00,  2.34it/s]
/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)
21 Jun 22:26    INFO  epoch 0 training [time: 6.08s, train loss: 2223.5039]
21 Jun 22:26    INFO  epoch 0 evaluating [time: 12.51s, valid_score: 0.000300]
21 Jun 22:26    INFO  valid result: 
ndcg@20 : 0.0003    recall@20 : 0.0006    mrr@20 : 0.0003
21 Jun 22:27    INFO  Saving current: saved/Pop-Jun-21-2026_22-26-40.pth
21 Jun 22:27    INFO  epoch 1 training [time: 5.42s, train loss: 2083.3063]
21 Jun 22:27    INFO  epoch 1 evaluating [time: 12.64s, valid_score: 0.000500]
2

In [12]:
print(f"\nBest valid score: {best_valid_score:.4f}")
print("Best valid result:")
for metric, score in best_valid_result.items():
    print(f"  {metric}: {score:.4f}")


Best valid score: 0.0030
Best valid result:
  ndcg@20: 0.0030
  recall@20: 0.0059
  mrr@20: 0.0032


## Evaluate on test set

In [13]:
test_result: dict[str, float] = trainer.evaluate(test_data)

print("Test results (Overall):")
for metric, value in test_result.items():
    print(f"  {metric}: {value:.4f}")

21 Jun 22:58    INFO  Loading model structure and parameters from saved/Pop-Jun-21-2026_22-26-40.pth


Test results (Overall):
  ndcg@20: 0.0024
  recall@20: 0.0046
  mrr@20: 0.0028


In [11]:
def evaluate_on_subset(
    data: AbstractDataLoader,
    mask: np.ndarray,
    label: str
):
    inter_feat = data.dataset.inter_feat
    cat_ds = data.dataset.copy(inter_feat[mask])
    cat_dl = FullSortEvalDataLoader(config, cat_ds, sampler=data._sampler)
    results = trainer.evaluate(cat_dl)
    print(f"\nEvaluation ({label})")
    print(f'-' * 20)
    print(f"  Interactions: {mask.sum()}")
    for metric, val in results.items():
        print(f"  {metric}: {val:.4f}")

# Map integer categories to labels
tok = dataset.field2token_id["category"]
CAT_LABELS = {tok["0"]: "warm", tok["1"]: "cold"}

uid_to_cat = dict(zip(
    dataset.user_feat[dataset.uid_field].numpy(),
    dataset.user_feat["category"].numpy(),
))

uid_array = test_data.dataset.inter_feat[dataset.uid_field].numpy()

for cat_id, cat_label in CAT_LABELS.items():
    cat_uids = {uid for uid, c in uid_to_cat.items() if c == cat_id}
    mask = np.isin(uid_array, list(cat_uids))
    if not mask.any():
        print(f"\n  {cat_label}: no users in test set — skipping")
        continue

    evaluate_on_subset(test_data, mask, cat_label)

21 Jun 22:58    INFO  Loading model structure and parameters from saved/Pop-Jun-21-2026_22-26-40.pth



Evaluation (warm)
--------------------
  Interactions: 46742
  ndcg@20: 0.0018
  recall@20: 0.0031
  mrr@20: 0.0030


21 Jun 22:58    INFO  Loading model structure and parameters from saved/Pop-Jun-21-2026_22-26-40.pth



Evaluation (cold)
--------------------
  Interactions: 150964
  ndcg@20: 0.0025
  recall@20: 0.0048
  mrr@20: 0.0028
